# 1. Analyse Exploratoire des Données

__Objectif Métier :__ Analyser le profil financier des emprunteurs afin de préparer les données pour un modèle de Credit Scoring (Prédiction de la Probabilité de Défaut).

__Jeu de données :__ [Give Me Some Credit de Kaggle](https://www.kaggle.com/competitions/GiveMeSomeCredit/data)

Ici, il s'agit essentiellement d'une première prise en main des données, puis du nettoyage de celles-ci (traitement des données aberrantes, manquantes...), nous pourrons ensuite observer leur distribution avant de passer au Feature engineering.

## a. Imports et premières observations
Dans un premier temps, on importe les bibliothèques utiles et le jeu de données et on assigne nos données à des variables.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

description_donnees = pd.read_excel("../data/raw/Data Dictionary.xls", index_col=1)
description_donnees.head()

On comprend que la variable cible, que l'on cherche à prédire, de notre étudie sera la variabe ```SeriousDlqin2yrs``` puisqu'on voit dans sa description "Person experienced 90 days past due delinquency or worse" ce qui correspond justement à la réponse de la question à laquelle on cherche à répondre : "Si je prête de l'argent à cette personne aujourd'hui, a-t-elle de fortes chances de cesser de me rembourser dans les 24 prochains mois ?"

In [ ]:
train_credit_data = pd.read_csv("../data/raw/cs-training.csv", index_col=0, header=0)
train_credit_data.head()

In [ ]:
test_credit_data = pd.read_csv("../data/raw/cs-test.csv", index_col=0, header=0)
test_credit_data.head() # Le fait que SeriousDlqin2yrs	soit en NaN dans la data de test confirme la conclusion précédente

## b. Traitement des Données Manquantes
La prochaine étape de notre étude va être de déterminer les variables qui comportent des données manquantes, on va ensuite discuter des hypothèses potentielles de ces manques et de comment nous allons les traiter.

In [ ]:
train_credit_data.info()

In [ ]:
compteur_nan = train_credit_data.isna().mean()
print(compteur_nan.sort_values(ascending=False)*100)


On tire plusieurs informations de notre appel à la méthode info() :
- On a un total de 150000 profils clients dans les données d'entraînement
- Parmi les variables, ```MonthlyIncome``` et ```NumberOfDependents``` sont celles qui présentent des données manquantes
    - Pour ```MonthlyIncome``` : On a 19.82% de valeurs manquantes
    - Pour ```NumberOfDependents``` : on a 2.61% de valeurs manquantes

 ## i. Variable : ```MonthlyIncome```
   - __Hypothèse :__ Ici, on peut deviner plusieurs raisons potentielles qui auraient poussé un client à laisser cette case vide :
       - Complexité : Le revenu du client est complexe et ne se résume pas en un monthly income
       - Précarité : c'est un revenu faible, le client ayant peur que cette statistique lui fasse défaut, il la cache
       - Irrégularité : son revenu mensuel est très volatile et le client ne sait pas quoi mettre
   - __Nature Statistique :__ Puisqu'on a pu identifier de nombreuses raisons potentielels d'omission volontaire, on conjecture que cette variable est de type MNAR (Missing not at random)
   - __Stratégie de traitement et validation :__ On va choisir d'utiliser une méthode d'imputation couplée à un indicateur binaire, ainsi, on peut à la fois combler le manque pour notre régression logistique et prendre en compte l'origine comportementale du trou. La distribution des revenus étant fortement étalée vers la droite, choisit la médiane comme stratégie étant donné la sensibilité de la moyenne aux valeurs extrêmes.

 ## ii. Variable : ```NumberOfDependents```
 - __Hypothèse :__ Cette variable indique le nombre de personnes à charge. On suppose qu'ici, une missing value peut être due au fait que le client suppose que laisser la case vide lors du remplissage est un 0 explicite.
 - __Nature Statistique :__ Ici aussi, on a probablement un cas de MNAR.
 - __Stratégie de traitement et validation :__ Si l'hypothèse est avérée, on va faire le choix de remplacer les NaN par des 0. Pour valider l'hypothèse, on va essayer de voir si $$\mathbb{P}(\text{SeriousDlqin2yrs}=1 \mid \text{NumberOfDependents}=\text{NaN}) \approx \mathbb{P}(\text{SeriousDlqin2yrs}=1 \mid \text{NumberOfDependents}=0)$$


### Validation de l'hypothèse pour ```NumberOfDependents```
On pose $P_1 = \mathbb{P}(\text{SeriousDlqin2yrs}=1 \mid \text{NumberOfDependents}=\text{NaN})$ et $P_2 =\mathbb{P}(\text{SeriousDlqin2yrs}=1 \mid \text{NumberOfDependents}=0)$

In [ ]:
masque = train_credit_data["NumberOfDependents"].isna()
train_data_nod_na = train_credit_data.loc[masque]

p_1 = train_data_nod_na["SeriousDlqin2yrs"].mean()

train_data_nod_0 = train_credit_data.loc[train_credit_data["NumberOfDependents"] == 0]
p_2 = train_data_nod_0["SeriousDlqin2yrs"].mean()

print(f'P_1 = {p_1}\nP_2 = {p_2}\n|P_1 - P_2| = {abs(p_1-p_2)*100}')


On trouve au final que $| P_1 - P_2 | \approx 1.3 \%$, on a donc un écart assez faible pour valider notre conjecture initiale pour cette variable, on va donc appliquer la stratégie de traitement qui va être de remplacer les NaN par des 0.

In [ ]:
train_credit_data["NumberOfDependents"] = train_credit_data["NumberOfDependents"].fillna(0)
train_credit_data.info()

### Mise en place de la stratégie d'imputation pour ```MonthlyIncome```
On fait le choix d'utiliser l'outil ```SimpleImputer``` de scikit learn. On fixe ```strategy = 'median'``` et ```add_indicator = True```.

In [ ]:
from sklearn.impute import SimpleImputer

imputer_income = SimpleImputer(strategy="median",add_indicator=True)
new_income = imputer_income.fit_transform(train_credit_data[["MonthlyIncome"]])

train_credit_data["MonthlyIncome"] = new_income[:,0]
train_credit_data["MissingIncomeFlag"] = new_income[:,1]

train_credit_data.info() #Plus aucune missing value
train_credit_data["MonthlyIncome"].describe()

On conclut qu'il n'y a plus aucune valeur manquante dans notre dataset, on peut donc passer au traitement des valeurs extrêmes.